In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
import json
import re
from utils import load_json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from matplotlib import pyplot as plt
import seaborn as sns

In [10]:
def importance_to_binary_label(importance):
    importance = int(importance)
    if importance in [1,2]:
        return 1
    elif importance in [3]:
        return 2   
    elif importance in [4]:
        return 3
    else:
        raise ValueError("Invalid importance value")

def importance_to_label(category):
    if "HIGH IMPORTANCE" in category:
        return 0
    elif "MEDIUM IMPORTANCE" in category:
        return 1
    elif "LOW IMPORTANCE" in category:
        return 2
    else:
        raise ValueError(f"Invalid category value: {category}")

def level_to_label(category):
    if "LEVEL 1" in category:
        return 1
    elif "LEVEL 2" in category:
        return 2
    elif "LEVEL 3" in category:
        return 3
    else:
        raise ValueError(f"Invalid category value: {category}")



def get_df(importances_path = './results/', levels_path = './results_levels/'):
    results = os.listdir(importances_path)

    data = []
    for result in results:
        #print(result)
        importances_result_data = load_json(os.path.join(importances_path, result))
        levels_result_data = load_json(os.path.join(levels_path, result))

        item_id = result.split("_")[-1].split(".")[0]

        importance_labels = importance_to_label(importances_result_data["cot_facts_law"]['classification'])
        levels_labels = level_to_label(levels_result_data["cot_facts_law"]['classification'])
        
        importance = int(levels_result_data["importance"])
        ground_truth = importance_to_binary_label(importance)
        data.append(
            {
                "item_id": item_id,
                "importance_labels": importance_labels,
                "levels_labels": levels_labels,
                "ground_truth": ground_truth,
                "importance": importance
            }
        )

    df = pd.DataFrame(data)
    return df


In [11]:
df = get_df('./results/', './results_levels/')

In [14]:
df.tail(20)

,item_id,importance_labels,levels_labels,ground_truth,importance
480,001-225042,0,1,2,3
481,001-201761,0,2,1,2
482,001-211018,0,2,1,2
483,001-229160,1,2,3,4
484,001-220885,0,1,2,3
485,001-177349,0,2,1,1
486,001-105624,1,2,1,2
487,001-227754,1,3,3,4
488,001-211125,0,1,1,1
489,001-205536,0,1,1,1


In [13]:
import pandas as pd
from scipy.stats import chi2_contingency

# Example dataframe
# df has columns: df['importance_labels'], df['levels_labels']
# each column can contain values in {1, 2, 3}

# 1. Create a contingency table
contingency_table = pd.crosstab(df['importance_labels'], df['levels_labels'])

# 2. Run Chi-square test of independence
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# 3. Print results
print("Contingency Table:")
print(contingency_table)
print("\nChi-square Statistic:", chi2)
print("Degrees of Freedom:", dof)
print("p-value:", p_value)

# 4. Interpret results
alpha = 0.05  # significance level
if p_value < alpha:
    print("\nResult: Reject H0")
    print("Conclusion: The two models produce significantly different predictions.")
else:
    print("\nResult: Fail to reject H0")
    print("Conclusion: No significant difference found between the two models.")

Contingency Table:
levels_labels        1    2   3
importance_labels              
0                  199  131  15
1                    4   50  37
2                    0   33  31

Chi-square Statistic: 188.33053612399056
Degrees of Freedom: 4
p-value: 1.2106599313232509e-39

Result: Reject H0
Conclusion: The two models produce significantly different predictions.
